# 1472. Design Browser History

**Difficulty:** Medium &nbsp;|&nbsp; **Topics:** design, linked-list, stack, array
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/design-browser-history/)

You have a **browser** of one tab where you start on the `homepage`, and you can
visit another URL, get back in the history a number of steps, or move forward in
the history a number of steps.

Implement the `BrowserHistory` class:

- `BrowserHistory(homepage)` initialises the object with the `homepage` of the
  browser.
- `visit(url)` visits `url` from the current page. **It clears up all the forward
  history.**
- `back(steps)` moves `steps` back in history. If you can only return `x` steps in
  the history and `steps > x`, you will return only `x` steps. Return the current
  URL after moving back.
- `forward(steps)` moves `steps` forward in history. If you can only forward `x`
  steps and `steps > x`, you will forward only `x` steps. Return the current URL
  after forwarding.

---

### Example

```
BrowserHistory b = new BrowserHistory("leetcode.com");
b.visit("google.com");        // leetcode.com -> google.com
b.visit("facebook.com");      // ... -> facebook.com
b.visit("youtube.com");       // ... -> youtube.com
b.back(1);                    // "facebook.com"
b.back(1);                    // "google.com"
b.forward(1);                 // "facebook.com"
b.visit("linkedin.com");      // forward history (youtube.com) is DESTROYED
b.forward(2);                 // "linkedin.com" - there is nothing forward of it
b.back(2);                    // "google.com"
b.back(7);                    // "leetcode.com" - clamped, only 1 step was possible
```

---

### Constraints

- `1 <= homepage.length, url.length <= 20`
- `1 <= steps <= 100`
- At most `5000` calls will be made to `visit`, `back` and `forward`

You use this class every day. It is also a doubly linked list wearing a costume -
and the fun of this problem is that two completely different structures, a linked
list and a pair of stacks, both model it exactly, and a plain array beats them both.

## Before you write anything

**1.** **One sentence in the statement is the whole problem: `visit` clears the
forward history.** Write down what that means for each of the three designs below
before you pick one:

```
an array + a cursor    ->  ?
two stacks             ->  ?
a doubly linked list   ->  ?
```

If your answer to any of them is "loop over the forward entries and delete them",
look again - all three can do it without a loop.

**2.** `back(steps)` **clamps**: asking for 7 steps when only 1 is possible returns
the page 1 step back, it does not raise and does not return the homepage twice. Write
the clamp for the array design in one line using `max`, and for the linked-list design
as a loop condition. Which one can get the fence-post wrong?

**3.** Trace the example by hand to the line `b.visit("linkedin.com")`, and write down
what happens to `youtube.com`. In the array design, is it *deleted*, or just
unreachable? Does the difference matter for correctness? For memory? (Those are two
different answers, and the second one is why real browsers are careful here.)

**4.** For the array design, `visit` is "truncate everything after the cursor, then
append". `del self.pages[i:]` is `O(n)` in the worst case. Is that a problem at 5000
calls? Say what the total cost of 5000 visits is - and notice this is the same
"amortised" argument you made in #933, in a different costume.

**5.** For the doubly linked list, `visit` is `cur.next = Node(url)` and
`node.prev = cur` - the old forward chain is not deleted, it simply becomes
unreachable and Python's garbage collector takes it. That is `O(1)` with no loop. Say
what the equivalent sentence is for the two-stack design, and which stack gets cleared.

**6.** All three are `O(1)`-ish and all three pass. So pick on something else: which
one would you rather **debug** at 3am, and which one would you rather extend when
someone asks for "show me the last 10 pages in a dropdown"? Answer both before you
write code.

## Two routes

**A - a list and a cursor** *(write this first)*

```
self.pages = [homepage]
self.i = 0
```

- `visit(url)`: `del self.pages[self.i + 1:]`, then append, then `self.i += 1`.
- `back(steps)`: `self.i = max(0, self.i - steps)`, return `self.pages[self.i]`.
- `forward(steps)`: `self.i = min(len(self.pages) - 1, self.i + steps)`, return it.

Three lines each, the clamps are one `max` and one `min`, and there is not a single
pointer to get wrong. This is the answer to submit, and honestly the answer to ship.

**B - a doubly linked list** *(the one that teaches something)*

Each page is a node with `url`, `prev` and `next`. `visit` builds a node, wires it to
`cur` in both directions, and moves `cur` onto it - which orphans the entire forward
chain in one assignment. `back`/`forward` walk `steps` times, stopping when `prev` or
`next` is `None`, which *is* the clamp.

`visit` is a true `O(1)` here rather than an amortised one, and `back(k)` is `O(k)`
rather than `O(1)`. Given `steps <= 100`, neither difference is measurable. What you
get instead is the thing worth having: you have now used the structure you built in
#707 route B to model something you actually use, and "the forward history is
whatever hangs off `cur.next`" is a sentence that explains the feature to a
non-programmer.

*(There is a third design - two stacks, back and forward - and `visit` clearing the
forward stack. It is the neatest of the three conceptually. Write it if you want the
set.)*

> **Same behaviour, three structures, and the simplest one wins.** This is the problem
> where you learn that "I could model this with a doubly linked list" is not an
> argument for doing it. Build A, submit A, then build B to see the shape - and be able
> to say why you shipped A.

In [ ]:
class BrowserHistory:

    def __init__(self, homepage: str):
        pass

    def visit(self, url: str) -> None:
        pass

    def back(self, steps: int) -> str:
        pass

    def forward(self, steps: int) -> str:
        pass

### The test harness

`back` and `forward` return the current URL, so most mistakes are visible. The one
that is not is `visit` failing to clear the forward history: nothing goes wrong until
somebody calls `forward`, which might be twenty calls later.

So `check` replays a call sequence against your class **and** against a model that is
a plain Python list plus an index - the obviously-correct version of route A. Every
`back` and `forward` return value is compared. Then, after **every** call including
every `visit`, it does the thing that catches the silent bug: it walks your browser
all the way forward, compares, walks all the way back, compares, and returns to where
it was. A forward history that should
have been destroyed shows up immediately, at the `visit` that failed to destroy it.

`stress` mixes visits and clamped jumps in both directions. Run this cell; don't edit
it.

In [ ]:
import random


def check(homepage, ops):
    '''Replay against BrowserHistory and a list+cursor model, probing both ends after every call.'''
    log = []
    try:
        bh = BrowserHistory(homepage)
    except Exception as e:
        return False, [f"   !! BrowserHistory({homepage!r}) raised {type(e).__name__}: {e}"]

    pages, i = [homepage], 0
    log.append(f"BrowserHistory({homepage!r})")

    def probe(after):
        '''Walk to both extremes and back, comparing - this is what catches a stale forward history.'''
        nonlocal i
        for direction, target in (("forward", len(pages) - 1), ("back", 0)):
            try:
                got = getattr(bh, direction)(len(pages))
            except Exception as e:
                log.append(f"   !! after {after}, {direction}({len(pages)}) raised {type(e).__name__}: {e}")
                return False
            if got != pages[target]:
                log.append(f"   !! after {after}, {direction}({len(pages)}) returned {got!r}")
                log.append(f"      the {'newest' if direction == 'forward' else 'oldest'} "
                           f"page should be {pages[target]!r}")
                log.append(f"      history is {pages}")
                return False
            i = target
        try:                                     # walk back to where we were
            got = bh.forward(i_saved)
        except Exception as e:
            log.append(f"   !! after {after}, restoring position raised {type(e).__name__}: {e}")
            return False
        i = min(len(pages) - 1, 0 + i_saved)
        if got != pages[i]:
            log.append(f"   !! after {after}, forward({i_saved}) from the start returned {got!r},"
                       f" should be {pages[i]!r}")
            return False
        return True

    for op, arg in ops:
        call = f"{op}({arg!r})"
        try:
            if op == "visit":
                bh.visit(arg)
                del pages[i + 1:]
                pages.append(arg)
                i += 1
                log.append(f"{call}            history {pages} cursor {i}")
            elif op == "back":
                got = bh.back(arg)
                i = max(0, i - arg)
                log.append(f"{call} -> {got!r}   (want {pages[i]!r})")
                if got != pages[i]:
                    log.append(f"   !! {call} must return {pages[i]!r}, got {got!r}")
                    log.append(f"      history {pages}, cursor now {i}")
                    return False, log
            else:
                got = bh.forward(arg)
                i = min(len(pages) - 1, i + arg)
                log.append(f"{call} -> {got!r}   (want {pages[i]!r})")
                if got != pages[i]:
                    log.append(f"   !! {call} must return {pages[i]!r}, got {got!r}")
                    log.append(f"      history {pages}, cursor now {i}")
                    return False, log
        except Exception as e:
            log.append(f"   !! {call} raised {type(e).__name__}: {e}")
            return False, log

        i_saved = i
        if not probe(call):
            return False, log

    return True, log


def stress(n, seed=0, max_steps=4):
    random.seed(seed)
    ops = []
    for k in range(n):
        r = random.random()
        if r < 0.4:
            ops.append(("visit", f"site{k}.com"))
        elif r < 0.7:
            ops.append(("back", random.randint(1, max_steps)))
        else:
            ops.append(("forward", random.randint(1, max_steps)))
    return check("home.com", ops)


def report(name, ok, log, tail=6):
    print(f"{'OK  ' if ok else 'FAIL'} {name}")
    if not ok:
        for line in log[-tail:]:
            print(f"       {line}")

In [ ]:
# tests
CASES = [
    ("the LeetCode example", "leetcode.com", [
        ("visit", "google.com"), ("visit", "facebook.com"), ("visit", "youtube.com"),
        ("back", 1), ("back", 1), ("forward", 1),
        ("visit", "linkedin.com"), ("forward", 2), ("back", 2), ("back", 7)]),

    ("nothing but the homepage", "a.com", [("back", 1), ("forward", 1), ("back", 100)]),

    ("question 2: back clamps at the homepage", "a.com", [
        ("visit", "b.com"), ("back", 100), ("back", 100), ("forward", 100)]),

    ("question 2: forward clamps at the newest page", "a.com", [
        ("visit", "b.com"), ("visit", "c.com"), ("forward", 100),
        ("back", 1), ("forward", 100)]),

    ("question 1: visit destroys the forward history", "a.com", [
        ("visit", "b.com"), ("visit", "c.com"), ("back", 2),
        ("visit", "d.com"), ("forward", 100)]),

    ("visit at the very start destroys everything ahead", "a.com", [
        ("visit", "b.com"), ("visit", "c.com"), ("visit", "d.com"),
        ("back", 100), ("visit", "z.com"), ("forward", 100), ("back", 100)]),

    ("back and forward all the way, repeatedly", "a.com", [
        ("visit", "b.com"), ("visit", "c.com"), ("visit", "d.com"),
        ("back", 3), ("forward", 3), ("back", 3), ("forward", 3), ("back", 1)]),

    ("the same url visited twice is two entries", "a.com", [
        ("visit", "b.com"), ("visit", "b.com"), ("back", 1), ("back", 1), ("forward", 2)]),

    ("steps of exactly 1 at every position", "a.com",
     [("visit", f"p{i}.com") for i in range(5)] + [("back", 1)] * 6 + [("forward", 1)] * 6),

    ("a long chain, then one visit near the start", "a.com",
     [("visit", f"p{i}.com") for i in range(100)] + [("back", 99), ("visit", "new.com"),
                                                      ("forward", 100), ("back", 100)]),
]

for name, home, ops in CASES:
    report(name, *check(home, ops))

for n, seed, ms in [(30, 1, 2), (100, 2, 4), (300, 3, 10), (1000, 4, 100)]:
    report(f"stress: {n} calls (seed {seed}, steps up to {ms})", *stress(n, seed, ms))

print("\ntrace of the LeetCode example:")
for line in check("leetcode.com", [
        ("visit", "google.com"), ("visit", "facebook.com"), ("visit", "youtube.com"),
        ("back", 1), ("back", 1), ("forward", 1),
        ("visit", "linkedin.com"), ("forward", 2), ("back", 2), ("back", 7)])[1]:
    print("  " + line)

## After it passes

- **Build all three.** Route A, the doubly linked list, and the two stacks - the same
  tests for each. Then write three sentences, one per design, each explaining what
  `visit` does to the forward history in that design. If a sentence needs a loop in it,
  you built that one wrong.
- **Measure the memory question from question 3.** In route A, visit 5000 pages, go
  back 4999, then visit one more. How many strings does `self.pages` still hold? Now
  the same in the linked-list version - what happened to the orphaned chain, and who
  freed it? (Then look up what a real browser does, because it does *not* free it
  immediately: that is what "the back button still shows the page instantly" costs.)
- **The invariant.** *`self.i` always points at a real page, and everything after it is
  reachable only by `forward`.* Write which line of `visit` could break it.
- **Make it real.** Real browsers have tabs (a `BrowserHistory` per tab), a bounded
  history (drop the oldest entries past 100 - which end of your list is that, and which
  design makes it cheap?), and `Ctrl+Shift+T` to reopen a closed tab - which means the
  history you just destroyed was not destroyed at all. Redesign `visit` to keep the
  forward chain somewhere recoverable, and notice you have just invented a second data
  structure to hold undo.
- Siblings: **#707 Design Linked List** (route B is your own class with a cursor on
  it), #155 Min Stack and #232 Implement Queue using Stacks (the two-stack design's
  family), #146 LRU Cache (a doubly linked list where the *order* is the data).